In [ ]:
import matplotlib.pyplot as plt
import netCDF4 as nc
import xarray as xr
import numpy as np
import cmocean
import os
import re

# VP 2020 Analysis
* Plot Hovmoller Diagram
* Compute front speed 

In [ ]:
import h5py

filename = "[folder name]]"
base_path = "[path to h5py folder]"
data_path = base_path + f'{filename}/data'

def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split('([0-9]+)', s)]
files = sorted([f for f in os.listdir(data_path) if f.endswith(".h5") and not f.startswith("._")], key=natural_sort_key)
all_h = []
all_t = []
                                                                                                                                   
for fname in files:
    with h5py.File(os.path.join(data_path, fname), 'r') as f:
        t = np.array(f['scales']['sim_time'])
        h = np.array(f['tasks']['h'])
        mid_y = h.shape[-1] // 2 
        h_equator = h[:, :, mid_y]
        
        all_t.append(t)
        all_h.append(h_equator)


""" 
Generate Hovmoller array
"""
time = np.concatenate(all_t)    
hovmoller = np.concatenate(all_h)   

Nx = hovmoller.shape[1]
lon = np.linspace(-5e6, 5e6, Nx)


In [ ]:
""" 
Plot Hovmoller diagram 
"""
time_days = time / (24 * 60 * 60)

plt.figure(figsize=(8, 12))
plt.pcolormesh(lon, time_days, hovmoller, cmap=cmocean.cm.balance_r, shading='auto')

plt.colorbar(shrink=0.4, label='h')
plt.xlabel('x [m]')
plt.ylabel('Time [days]')
plt.ylim(time_days[0], time_days[-1])

plt.title('$N_x, N_y = 256, \gamma = 5$', fontsize=14)
plt.savefig(base_path + 'hovmoller.png', dpi =200, bbox_inches = 'tight')
plt.show()

In [ ]:

from scipy.stats import linregress

""" 
Obtain approximate slope on Hovmoller by linear regression
    * In chosen time range
    * NOTE: not useful when noise is present --> use next cell
"""

t_mask = (time_days >= 195) & (time_days <= 200)
time_window = time_days[t_mask]
hov_window = hovmoller[t_mask, :]

peak_indices = np.argmax(hov_window, axis=1)
peak_x_positions = lon[peak_indices]

slope, intercept, r_value, p_value, std_err = linregress(time_window, peak_x_positions)
speed_m_s = slope / (24 * 60 * 60)

print(f"v= {speed_m_s:.2f} m/s")
print(f"R^2 = {r_value**2:.4f}")

plt.figure(figsize=(8, 6))
plt.pcolormesh(lon, time_window, hov_window, cmap=cmocean.cm.balance_r, shading='auto')
plt.plot(peak_x_positions, time_window, 'k.', label='Peak $h$')
plt.plot(intercept + slope * time_window, time_window, 'r--', linewidth=2, label=f'$v \\approx$ {speed_m_s:.2f} m/s')
plt.xlabel('x [m]')
plt.ylabel('Time [days]')
plt.legend()
plt.savefig(base_path + '/hovmoller_slope_linregress.png', dpi = 200, bbox_inches = 'tight')
plt.show()

In [ ]:
""" 
Select two points on Hovmoller along front
    * Returns slope of plotted line
"""
%matplotlib tk

plt.figure(figsize=(8, 12))
plt.pcolormesh(lon, time_days, hovmoller, cmap=cmocean.cm.balance_r, shading='auto')
points = plt.ginput(2) 
x1, t1_days = points[0]
x2, t2_days = points[1]
dx = x2 - x1
dt_days = t2_days - t1_days
dt_seconds = dt_days * 24 * 60 * 60

speed_m_s = dx / dt_seconds
print(f"Calculated Front Speed : {speed_m_s:.4f} m/s")
plt.plot([x1, x2], [t1_days, t2_days], 'k--', linewidth=3)
plt.show()